In [1]:
import os
import psycopg2
from dotenv import load_dotenv

# Ensure system-level or updated .env variables take precedence over existing environment variables
load_dotenv(override=True)

# Database configuration loaded from environment variables for security and flexibility
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Initialize connection reference to ensure it is available in the 'finally' safety block
conn = None

try:
    # --- 1. Database Connection Establishment ---
    print(f"Connecting to {HOST}...")
    
    conn = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )
    
    # Cursor is required to execute queries and manage the context of the database operations
    cur = conn.cursor()

    # --- 2. Schema Definition and Execution ---
    print("Creating 'price_history' table...")
    
    # Using 'IF NOT EXISTS' to prevent runtime errors if the script is executed multiple times
    # High precision DECIMAL scales (e.g., 16, 8) prevent rounding errors in quantitative risk/momentum metrics
    # ON DELETE CASCADE ensures referential integrity without leaving orphaned historical data
    cur.execute("""
        CREATE TABLE IF NOT EXISTS public.price_history (
            ticker TEXT NOT NULL,
            date DATE NOT NULL,

            open_price DECIMAL(16, 4),
            high_price DECIMAL(16, 4),
            low_price DECIMAL(16, 4),
            close_price DECIMAL(16, 4),
            volume BIGINT,

            return_3m DECIMAL(16, 8),
            return_6m DECIMAL(16, 8),
            momentum_score DECIMAL(16, 8),
            returns DECIMAL(16, 8),
            volatility DECIMAL(16, 8),
            low_vol_score DECIMAL(16, 8),

            PRIMARY KEY (ticker, date),

            CONSTRAINT fk_assets 
                FOREIGN KEY (ticker) 
                REFERENCES public.assets(ticker) 
                ON DELETE CASCADE
        );
    """)
    
    # --- 3. Transaction Management ---
    # Explicitly commit the DDL transaction to persist the table creation in PostgreSQL
    conn.commit()
    print("Price history table created successfully.")

except Exception as e:
    # Catch-all block to prevent script crashes and log the specific PostgreSQL/Connection exception
    print(f"Database error: {e}")

finally:
    # --- 4. Resource Cleanup ---
    # Safe teardown of database objects to prevent connection leaks and idle session buildup
    if conn:
        cur.close()
        conn.close()
        print("Connection closed.")

Connecting to aws-1-eu-north-1.pooler.supabase.com...
Creating 'price_history' table...
Price history table created successfully.
Connection closed.
